In [3]:
import pandas as pd
df = pd.read_csv("ResaleflatpricesbasedonregistrationdatefromJan2017onwards.csv")
df.head()

,month,town,flat_type,block,street_name,storey_range,floor_area_sqm,flat_model,lease_commence_date,remaining_lease,resale_price
0,2017-01,ANG MO KIO,2 ROOM,406,ANG MO KIO AVE 10,10 TO 12,44.0,Improved,1979,61 years 04 months,232000.0
1,2017-01,ANG MO KIO,3 ROOM,108,ANG MO KIO AVE 4,01 TO 03,67.0,New Generation,1978,60 years 07 months,250000.0
2,2017-01,ANG MO KIO,3 ROOM,602,ANG MO KIO AVE 5,01 TO 03,67.0,New Generation,1980,62 years 05 months,262000.0
3,2017-01,ANG MO KIO,3 ROOM,465,ANG MO KIO AVE 10,04 TO 06,68.0,New Generation,1980,62 years 01 month,265000.0
4,2017-01,ANG MO KIO,3 ROOM,601,ANG MO KIO AVE 5,01 TO 03,67.0,New Generation,1980,62 years 05 months,265000.0


In [4]:
print(df.shape)
print(df.dtypes)

(228225, 11)
month                      str
town                       str
flat_type                  str
block                      str
street_name                str
storey_range               str
floor_area_sqm         float64
flat_model                 str
lease_commence_date      int64
remaining_lease            str
resale_price           float64
dtype: object


In [5]:
df['remaining_lease'].unique()[:10]

<ArrowStringArray>
['61 years 04 months', '60 years 07 months', '62 years 05 months',
  '62 years 01 month',           '63 years', '61 years 06 months',
 '58 years 04 months', '59 years 08 months', '59 years 06 months',
           '60 years']
Length: 10, dtype: str

In [7]:
import re
def parse_remaining_lease(lease_str):

    years = 0

    months = 0

    

    year_match = re.search(r'(\d+) year', lease_str)

    if year_match:

        years = int(year_match.group(1))


    month_match = re.search(r'(\d+) months', lease_str)

    if month_match:
        months = int(month_match.group(1))

    

    return years + months/12

In [17]:
parse_remaining_lease("61 years 04 months")

61.333333333333336

In [18]:
df['remaining_lease'] = df['remaining_lease'].apply(parse_remaining_lease)

In [56]:
print(df.dtypes)

town                       str
flat_type                  str
storey_range           float64
floor_area_sqm         float64
flat_model                 str
lease_commence_date      int64
remaining_lease        float64
resale_price           float64
year                     int64
dtype: object


In [20]:
df['storey_range'].unique()[:10]

<ArrowStringArray>
['10 TO 12', '01 TO 03', '04 TO 06', '07 TO 09', '13 TO 15', '19 TO 21',
 '22 TO 24', '16 TO 18', '34 TO 36', '28 TO 30']
Length: 10, dtype: str

In [47]:

def parse_storey_range(storey_str):

    top = 0

    low = 0

    

    top_floor = re.search(r'TO (\d+)', storey_str)

    if top_floor:

        top = int(top_floor.group(1))


    lower_floor = re.search(r'(\d+) TO', storey_str)

    if lower_floor:

       low = int(lower_floor.group(1))


    

    return (top+low)/2

In [48]:
parse_storey_range('10 TO 12')

11.0

In [49]:
df['storey_range'] = df['storey_range'].apply(parse_storey_range)

In [52]:
df = df.drop(columns=['block', 'street_name'])

In [53]:
df['year'] = df['month'].str[:4].astype(int)

df = df.drop(columns=['month'])

In [55]:
df['year'] = df['month'].str[:4].astype(int)
df['month_num'] = df['month'].str[5:].astype(int)
df = df.drop(columns=['month'])

KeyError: 'month'

In [57]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()

df['town'] = le.fit_transform(df['town'])
df['flat_type'] = le.fit_transform(df['flat_type'])
df['flat_model'] = le.fit_transform(df['flat_model'])

In [58]:
print(df.dtypes)

town                     int64
flat_type                int64
storey_range           float64
floor_area_sqm         float64
flat_model               int64
lease_commence_date      int64
remaining_lease        float64
resale_price           float64
year                     int64
dtype: object


In [59]:
X = df.drop(columns=['resale_price'])  # everything except price
y = df['resale_price']                  # just the price

In [60]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(X_train.shape)
print(X_test.shape)

(182580, 8)
(45645, 8)


In [61]:
from sklearn.ensemble import RandomForestRegressor

model = RandomForestRegressor(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",100
,"criterion criterion: {""squared_error"", ""absolute_error"", ""friedman_mse"", ""poisson""}, default=""squared_error""The function to measure the quality of a split. Supported criteriaare ""squared_error"" for the mean squared error, which is equal tovariance reduction as feature selection criterion and minimizes the L2loss using the mean of each terminal node, ""friedman_mse"", which usesmean squared error with Friedman's improvement score for potentialsplits, ""absolute_error"" for the mean absolute error, which minimizesthe L1 loss using the median of each terminal node, and ""poisson"" whichuses reduction in Poisson deviance to find splits.Training using ""absolute_error"" is significantly slowerthan when using ""squared_error""... versionadded:: 0.18 Mean Absolute Error (MAE) criterion... versionadded:: 1.0 Poisson criterion.",'squared_error'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",None
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=1.0The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None or 1.0, then `max_features=n_features`... note:: The default of 1.0 is equivalent to bagged trees and more randomness can be achieved by setting smaller values, e.g. 0.3... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to 1.0.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",1.0
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsample

In [62]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

y_pred = model.predict(X_test)

mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print(f"MAE: {mae:,.0f}")
print(f"RMSE: {rmse:,.0f}")
print(f"R2: {r2:.4f}")

MAE: 26,207
RMSE: 38,327
R2: 0.9587


In [64]:
import pickle

with open("model.pkl", "wb") as f:
    pickle.dump(model, f)

In [63]:
# We need to retrain the encoders and save them
# First reload the original data and re-encode, saving each encoder

from sklearn.preprocessing import LabelEncoder
import pickle

# Reload original data
df_orig = pd.read_csv("ResaleflatpricesbasedonregistrationdatefromJan2017onwards.csv")

le_town = LabelEncoder()
le_flat_type = LabelEncoder()
le_flat_model = LabelEncoder()

le_town.fit(df_orig['town'])
le_flat_type.fit(df_orig['flat_type'])
le_flat_model.fit(df_orig['flat_model'])

with open("le_town.pkl", "wb") as f:
    pickle.dump(le_town, f)
with open("le_flat_type.pkl", "wb") as f:
    pickle.dump(le_flat_type, f)
with open("le_flat_model.pkl", "wb") as f:
    pickle.dump(le_flat_model, f)

print("All saved.")

All saved.
